# Grid F — the 61 higher-order runs that never ran

`results/tnn` holds 150 of its 180 planned runs and `results/tnn_mu0` 149 of 180.
Every one of the 61 missing runs is the **`A8` arm** — `s5 = tnn`, the only
configuration in the paper that message-passes over a real cell complex — on
**DHFR, IMDB-BINARY and PROTEINS** (plus one ENZYMES fold at `mu = 0`).

## Why they are missing, which is not why it looked like they were

Not a crash and not a hang. The Modal orchestrator ran with
`DMD_STALL_TIMEOUT_S = 420` and killed any batch that went seven minutes without
output. On those three datasets the `A3` arm — same `cycle_basis` proposal, one
stage apart from `A8` — takes a **median 470–700 s and a maximum of 1497 s** per
run. The runs were killed for being slow, not for being stuck, and the arm that
was dropped is the expensive one by construction.

So this notebook's job is not to fix a bug. It is to give 61 slow runs enough
time, with a timeout set per *run* rather than per *batch*, so that one slow run
can never take four good ones down with it.

## What it costs, measured rather than guessed

Sizing the last Kaggle notebook, I assumed a T4 was ~2.5x slower than the Modal
L4 and cut the plan accordingly. That was wrong, and the seed-1/2 runs measured
it: over 36 matched (dataset, gamma) cells, **median T4/L4 = 0.97**. This
workload is bound by Python and PyG overhead on small graphs, not by the GPU.

Estimating each missing `A8` run from the `A3` run beside it (median `A8/A3`
factor over the arms where both ran: **0.93**):

| grid | dataset | n | est. per run | est. total |
|---|---|---:|---:|---:|
| `tnn` | DHFR | 10 | 57 s | 0.16 h |
| `tnn` | PROTEINS | 10 | 506 s | 1.41 h |
| `tnn` | IMDB-BINARY | 10 | 601 s | 1.67 h |
| `tnn_mu0` | DHFR | 10 | 56 s | 0.16 h |
| `tnn_mu0` | ENZYMES | 1 | 232 s | 0.06 h |
| `tnn_mu0` | PROTEINS | 10 | 436 s | 1.21 h |
| `tnn_mu0` | IMDB-BINARY | 10 | 646 s | 1.80 h |

**~6.5 GPU-h**, ~8.4 h with a 1.3x margin. It fits one 12 h session — and Kaggle
gives **two T4s**, which the notebook uses in parallel, so expect ~3.5 h wall for
the same quota cost. If it does not finish, it resumes: nothing already stored is
re-run.

## Design

- **One run per subprocess.** ~30 s of import overhead each (~9 % of the job) in
  exchange for a timeout that can only ever cost the single run that hit it. This
  grid has already been lost once to a batch-level timeout.
- **Two workers, two GPUs, two private stores**, merged at the end. `ResultsStore`
  appends to one CSV; two processes appending to the same file would interleave
  rows, so each worker writes its own and they are merged through
  `tools/merge_results.py`, which refuses a duplicate `run_id`.
- **The plan is rebuilt by the repo's own `build_plan`**, then intersected with a
  literal list of the 61 missing `run_id`s. If the rebuilt plan does not contain
  every one of them, the notebook stops: that means the config drifted and the
  new runs would not be comparable with the stored ones.
- **The smoke test runs `A8`**, not `A0`. `topomodelx` / `toponetx` import
  cleanly in most environments and then fail inside the CWN layer; two minutes
  here is cheaper than finding out at run 40.

## Kaggle settings

`Accelerator = GPU T4 x2`, `Internet = On` (TUDataset downloads on first use).
`Persistence` off. Run all cells.

In [ ]:
# =====================================================================
# CONFIG - every knob of this notebook is in this cell.
# =====================================================================
REPO_URL = "https://github.com/AlGoRythm3000/Differentiable-Motif-Discovery.git"
BRANCH   = "main"
REPO_DIR = "/kaggle/working/repo"
OUT_DIR  = "/kaggle/working/results"
DATA_DIR = "/kaggle/working/datasets"

DATASETS = ["synthetic_bottleneck", "MUTAG", "PROTEINS", "IMDB-BINARY", "ENZYMES", "DHFR"]
CV_FOLDS = 5
SEEDS    = [0]           # Grid F is a single-seed grid; these are its OWN missing runs.
GAMMAS   = [0.0, 0.03]

# The two halves of Grid F differ only in mu, and mu is what decides whether the
# rank-2 level is empty. `sparsity_weights=None` keeps RunSpec.sparsity_weight
# unset so the run_id has no `_m` segment - which is how the mu=0.05 half was
# stored, and the run_ids have to match byte for byte or nothing resumes.
GRIDS = {
    "tnn":     {"sparsity_weight": 0.05, "sparsity_weights": None},
    "tnn_mu0": {"sparsity_weight": 0.05, "sparsity_weights": [0.0]},
}

# --- parallelism -----------------------------------------------------
# None = one worker per visible GPU. Kaggle's "T4 x2" gives two, and these runs
# are independent, so two workers halve the wall clock at the same quota cost.
NUM_WORKERS = None

# --- session limits --------------------------------------------------
WALL_BUDGET_H = 11.0      # Kaggle kills a GPU session at 12 h; stop before that.
# Per RUN, not per batch. The longest A3 run on these datasets was 1497 s, so
# 45 min is a ~1.8x margin over the slowest thing we have ever seen here, and a
# run that blows it costs exactly itself.
PER_RUN_TIMEOUT_S = 45 * 60

# Attach a previous session's output as an input dataset and point this at the
# directory holding its `tnn/` and `tnn_mu0/` subdirectories.
RESUME_FROM = None        # e.g. "/kaggle/input/dmd-gridf-session1"

INSTALL_DEPS = True

In [ ]:
# =====================================================================
# SETUP - dependencies, clone at the pinned branch
# =====================================================================
import os, subprocess, sys

if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch_geometric", "networkx", "pyyaml"], check=False)
    # topomodelx/toponetx pin numpy<2 and pull `pyg-nightly`, a SECOND
    # distribution of the `torch_geometric` package. Letting their resolver run
    # mixes files from both distributions in site-packages and produces
    # "partially initialized module 'torch_geometric' has no attribute 'typing'"
    # at import time. --no-deps installs them without touching PyG or numpy.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                    "topomodelx", "toponetx"], check=False)

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH],
                   check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())

# Every run in this notebook is the A8 arm, and A8 is the only arm that message-
# passes through TopoModelX. A `check=False` install that quietly failed would
# otherwise surface as 61 identical failures an hour from now.
#
# The symbol, not the package: `models/message_passing.py` imports exactly this
# one class, and builds the incidence matrices in torch rather than through
# `toponetx.classes.CellComplex` (whose import chain pulls pyarrow/trimesh). So
# `toponetx` is installed for completeness but never imported, and checking it
# here would fail the notebook over a dependency no run touches.
try:
    from topomodelx.nn.cell.cwn_layer import CWNLayer  # noqa: F401
    print("  topomodelx.CWNLayer: ok")
except Exception as exc:
    raise SystemExit(f"topomodelx's CWNLayer does not import ({exc}). Every run "
                     "here is the A8 (s5=tnn) arm and goes through it; fix the "
                     "install before going further.")

In [ ]:
# =====================================================================
# ENVIRONMENT LOG
# =====================================================================
import torch

from tools.experiment_grid import GridConfig, build_plan, environment_info
from tools.results_store import ResultsStore

ENV = environment_info(REPO_DIR)
ENV.update({"branch": BRANCH, "runner": "kaggle", "grid": "F (A8 backfill)",
            "seeds": ",".join(str(s) for s in SEEDS)})
for key, value in ENV.items():
    print(f"{key:>18}: {value}")

N_GPU = torch.cuda.device_count()
DEVICE = "cuda" if N_GPU else "cpu"
print(f"{'device':>18}: {DEVICE}")
print(f"{'gpus':>18}: {N_GPU}")
for i in range(N_GPU):
    cap = torch.cuda.get_device_capability(i)
    print(f"{'gpu ' + str(i):>18}: {torch.cuda.get_device_name(i)} (sm_{cap[0]}{cap[1]})")
if N_GPU:
    # A wheel with no kernels for this architecture makes CUDA JIT from PTX at
    # every first launch, which is one of the ways a run turns from 40 s into
    # hours. Worth knowing before the smoke test, not after.
    print(f"{'built for':>18}: {', '.join(torch.cuda.get_arch_list())}")
else:
    print("WARNING: no GPU visible - set Settings -> Accelerator to GPU T4 x2.")

In [ ]:
# =====================================================================
# SMOKE TEST - one real A8 run, on the arm every run here uses.
#
# A8 is `s5 = tnn`: message passing over a CWN cell complex rather than over
# flattened star edges. It is the only arm in the paper that exercises
# topomodelx, and the two libraries import cleanly in environments where the
# CWN layer then fails. MUTAG at gamma=0.03 took 39 s on Modal, so this cell
# costs about a minute and answers the only question that can waste the session.
# =====================================================================
import time

from tools.experiment_grid import RunSpec, load_arm, run_single
from tools.config_loader import load_all_configs

ALL_CONFIGS = load_all_configs(f"{REPO_DIR}/configs")
SMOKE = {}
for dev in (["cuda", "cpu"] if N_GPU else ["cpu"]):
    cfg = GridConfig(datasets=["MUTAG"], configs_dir=f"{REPO_DIR}/configs",
                     seeds=SEEDS, cv_folds=CV_FOLDS, grad_clip=1.0,
                     sparsity_weight=0.05, data_root=DATA_DIR, device=dev,
                     time_budget_s=None)
    cfg.commit_sha = ENV.get("commit_sha", "unknown")
    ds, nfeat, nclass, _ = load_arm("MUTAG", cfg)
    spec = RunSpec("A", "A8", "MUTAG", 0.03, SEEDS[0], fold=0, proxy="r_bar")
    t0 = time.time()
    # `run_single` RAISES on failure rather than returning a status, so the
    # traceback is the diagnosis and must not be swallowed.
    try:
        result = run_single(spec, cfg, ALL_CONFIGS["A8"], ds, nfeat, nclass)
    except Exception as exc:
        import traceback
        traceback.print_exc()
        raise SystemExit(f"the A8 arm does not run on {dev}: {exc}")
    SMOKE[dev] = time.time() - t0
    row = result["row"]
    print(f"  {dev:>6}: {SMOKE[dev]:6.1f}s   acc={row['test_acc']}  "
          f"cells={row['num_cells']}  status={row['status']}")

# Not a warning to scroll past: at this ratio the GPU plan cannot finish, and
# the CPU is the faster machine for this workload.
if SMOKE.get("cuda", 0) > 3 * SMOKE.get("cpu", 1e9):
    print(f"\nGPU is {SMOKE['cuda'] / SMOKE['cpu']:.1f}x SLOWER than CPU. "
          f"Falling back to CPU, one worker.")
    DEVICE, N_GPU = "cpu", 0
print(f"\nrunning on: {DEVICE}")

In [ ]:
# =====================================================================
# PLAN - rebuild both halves of Grid F, then keep only the missing runs.
#
# The run_ids below are the ones absent from results/tnn and results/tnn_mu0 as
# of commit 9658349. They are not used to CONSTRUCT the specs - build_plan does
# that, so a spec is built by the same code that built its stored twins and the
# run_id is byte-identical - only to SELECT them. If the rebuilt plan is missing
# any of them, the grid definition has drifted and the new runs would not be
# comparable with the stored ones, so the notebook stops.
# =====================================================================
from collections import Counter

MISSING = {
    "tnn": [
        "A_A8_DHFR_g0.0_s0_f0", "A_A8_DHFR_g0.0_s0_f1", "A_A8_DHFR_g0.0_s0_f2",
        "A_A8_DHFR_g0.0_s0_f3", "A_A8_DHFR_g0.0_s0_f4",
        "A_A8_DHFR_g0.03_s0_f0", "A_A8_DHFR_g0.03_s0_f1", "A_A8_DHFR_g0.03_s0_f2",
        "A_A8_DHFR_g0.03_s0_f3", "A_A8_DHFR_g0.03_s0_f4",
        "A_A8_IMDB-BINARY_g0.0_s0_f0", "A_A8_IMDB-BINARY_g0.0_s0_f1",
        "A_A8_IMDB-BINARY_g0.0_s0_f2", "A_A8_IMDB-BINARY_g0.0_s0_f3",
        "A_A8_IMDB-BINARY_g0.0_s0_f4",
        "A_A8_IMDB-BINARY_g0.03_s0_f0", "A_A8_IMDB-BINARY_g0.03_s0_f1",
        "A_A8_IMDB-BINARY_g0.03_s0_f2", "A_A8_IMDB-BINARY_g0.03_s0_f3",
        "A_A8_IMDB-BINARY_g0.03_s0_f4",
        "A_A8_PROTEINS_g0.0_s0_f0", "A_A8_PROTEINS_g0.0_s0_f1",
        "A_A8_PROTEINS_g0.0_s0_f2", "A_A8_PROTEINS_g0.0_s0_f3",
        "A_A8_PROTEINS_g0.0_s0_f4",
        "A_A8_PROTEINS_g0.03_s0_f0", "A_A8_PROTEINS_g0.03_s0_f1",
        "A_A8_PROTEINS_g0.03_s0_f2", "A_A8_PROTEINS_g0.03_s0_f3",
        "A_A8_PROTEINS_g0.03_s0_f4",
    ],
    "tnn_mu0": [
        "A_A8_DHFR_g0.0_m0.0_s0_f0", "A_A8_DHFR_g0.0_m0.0_s0_f1",
        "A_A8_DHFR_g0.0_m0.0_s0_f2", "A_A8_DHFR_g0.0_m0.0_s0_f3",
        "A_A8_DHFR_g0.0_m0.0_s0_f4",
        "A_A8_DHFR_g0.03_m0.0_s0_f0", "A_A8_DHFR_g0.03_m0.0_s0_f1",
        "A_A8_DHFR_g0.03_m0.0_s0_f2", "A_A8_DHFR_g0.03_m0.0_s0_f3",
        "A_A8_DHFR_g0.03_m0.0_s0_f4",
        "A_A8_ENZYMES_g0.03_m0.0_s0_f0",
        "A_A8_IMDB-BINARY_g0.0_m0.0_s0_f0", "A_A8_IMDB-BINARY_g0.0_m0.0_s0_f1",
        "A_A8_IMDB-BINARY_g0.0_m0.0_s0_f2", "A_A8_IMDB-BINARY_g0.0_m0.0_s0_f3",
        "A_A8_IMDB-BINARY_g0.0_m0.0_s0_f4",
        "A_A8_IMDB-BINARY_g0.03_m0.0_s0_f0", "A_A8_IMDB-BINARY_g0.03_m0.0_s0_f1",
        "A_A8_IMDB-BINARY_g0.03_m0.0_s0_f2", "A_A8_IMDB-BINARY_g0.03_m0.0_s0_f3",
        "A_A8_IMDB-BINARY_g0.03_m0.0_s0_f4",
        "A_A8_PROTEINS_g0.0_m0.0_s0_f0", "A_A8_PROTEINS_g0.0_m0.0_s0_f1",
        "A_A8_PROTEINS_g0.0_m0.0_s0_f2", "A_A8_PROTEINS_g0.0_m0.0_s0_f3",
        "A_A8_PROTEINS_g0.0_m0.0_s0_f4",
        "A_A8_PROTEINS_g0.03_m0.0_s0_f0", "A_A8_PROTEINS_g0.03_m0.0_s0_f1",
        "A_A8_PROTEINS_g0.03_m0.0_s0_f2", "A_A8_PROTEINS_g0.03_m0.0_s0_f3",
        "A_A8_PROTEINS_g0.03_m0.0_s0_f4",
    ],
}

COMMON = dict(datasets=DATASETS, configs_dir=f"{REPO_DIR}/configs",
              seeds=SEEDS, cv_folds=CV_FOLDS, grad_clip=1.0, gammas=GAMMAS,
              data_root=DATA_DIR, device=DEVICE,
              # Left None on purpose: the projection-based trimmer drops whole
              # tiers on one global mean seconds-per-run, and this plan is
              # deliberately all-expensive, which is exactly the input that
              # makes it delete everything. The wall-clock check in the run cell
              # is the cutoff instead - it stops, and the next session resumes.
              time_budget_s=None)

CONFIGS, PLAN = {}, []
for grid, mu in GRIDS.items():
    cfg = GridConfig(**COMMON, **mu)
    cfg.commit_sha = ENV.get("commit_sha", "unknown")
    CONFIGS[grid] = cfg
    by_id = {s.run_id: s for s in build_plan(cfg) if s.config_id in ("A0", "A3", "A8")}
    absent = [r for r in MISSING[grid] if r not in by_id]
    if absent:
        raise SystemExit(
            f"{grid}: {len(absent)} of the missing run_ids are not in the rebuilt "
            f"plan, e.g. {absent[:3]}.\nThe grid definition has drifted since "
            "these were planned; do not run - the results would not be comparable.")
    PLAN += [(grid, by_id[r]) for r in MISSING[grid]]

print(f"{len(PLAN)} runs to do")
for (grid, ds), n in sorted(Counter((g, s.dataset) for g, s in PLAN).items()):
    print(f"   {grid:<9}{ds:<22}{n}")

In [ ]:
# =====================================================================
# RESUME + QUARANTINE + WORKER ASSIGNMENT
#
# Each worker gets its OWN store per grid. ResultsStore appends to a single
# runs.csv; two processes appending to the same file interleave rows and produce
# a CSV no check can repair. They are merged at the end, through a merger that
# refuses a duplicate run_id.
# =====================================================================
import json, shutil
from pathlib import Path

WORKERS = NUM_WORKERS or max(1, N_GPU)
QUARANTINE = Path(OUT_DIR) / "quarantine.json"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

def store_dir(worker, grid):
    return f"{OUT_DIR}/w{worker}/{grid}"

# Anything already stored - in a previous session's output, or in this session's
# own working directory after a restart - is skipped rather than re-run.
done = set()
for root in ([Path(RESUME_FROM)] if RESUME_FROM else []) + [Path(OUT_DIR)]:
    for runs in root.rglob("runs.csv"):
        with open(runs, newline="") as f:
            import csv as _csv
            done |= {r["run_id"] for r in _csv.DictReader(f)}
if RESUME_FROM:
    for grid in GRIDS:
        src = Path(RESUME_FROM) / grid
        if src.is_dir():
            dst = Path(store_dir(0, grid))
            if not dst.exists():
                shutil.copytree(src, dst)
                print(f"resumed {grid} from {src}")
print(f"already stored : {len(done)}")

# A run that wedged a previous session is retried once, then left alone: a
# pathological spec that is retried forever eats every session in turn.
quarantined = json.loads(QUARANTINE.read_text()) if QUARANTINE.exists() else {}
blocked = {k for k, n in quarantined.items() if n >= 2}

todo = [(g, s) for g, s in PLAN if s.run_id not in done and s.run_id not in blocked]
if blocked:
    print(f"quarantined    : {len(blocked)} (timed out twice)")
print(f"to run         : {len(todo)}")

# Longest-processing-time-first onto the least-loaded worker. Balancing the
# makespan matters more than leaving whole cheap runs behind, because the
# estimate (~6.5 GPU-h over two workers) fits the session with room to spare.
# Costs are the A3 median on the same dataset scaled by the measured A8/A3
# factor of 0.93 - a planning hint only, nothing depends on its accuracy.
COST_S = {"DHFR": 60, "MUTAG": 40, "synthetic_bottleneck": 40,
          "ENZYMES": 240, "PROTEINS": 480, "IMDB-BINARY": 620}
todo.sort(key=lambda gs: (-COST_S.get(gs[1].dataset, 300), gs[1].run_id))
ASSIGNED = [[] for _ in range(WORKERS)]
load = [0.0] * WORKERS
for grid, spec in todo:
    w = load.index(min(load))
    ASSIGNED[w].append((grid, spec))
    load[w] += COST_S.get(spec.dataset, 300)
for w in range(WORKERS):
    print(f"   worker {w}: {len(ASSIGNED[w]):>3} runs, ~{load[w] / 3600:.2f} h estimated")
print(f"   estimated wall: ~{max(load) / 3600:.2f} h on {WORKERS} worker(s)")

In [ ]:
# =====================================================================
# WORKER - ONE run, in its own process.
#
# A subprocess rather than a thread because a wedged CUDA call cannot be
# interrupted from Python: only killing the process gets the session back. One
# run per process rather than a batch because that is the difference between a
# timeout costing one run and a timeout costing the four good runs beside it -
# which is precisely how these 61 runs were lost in the first place.
#
# Everything crosses the boundary as plain JSON, so the parent never holds a
# CUDA context and can always kill the child.
# =====================================================================
WORKER = """
import json, sys
sys.path.insert(0, sys.argv[1])
from tools.experiment_grid import GridConfig, RunSpec, run_grid
from tools.results_store import ResultsStore

payload = json.load(open(sys.argv[2]))
config = GridConfig(**payload["config"])
config.commit_sha = payload["commit_sha"]
plan = [RunSpec(**s) for s in payload["specs"]]
run_grid(config, ResultsStore(payload["out_dir"]), plan=plan, verbose=True)
"""
Path(f"{OUT_DIR}/_worker.py").write_text(WORKER)

def config_kwargs(grid):
    """The GridConfig the child rebuilds, as JSON. Must match CONFIGS[grid]."""
    kwargs = dict(COMMON, **GRIDS[grid])
    kwargs["device"] = DEVICE
    return kwargs

def spec_json(spec):
    return {"tier": spec.tier, "config_id": spec.config_id, "dataset": spec.dataset,
            "gamma": spec.gamma, "seed": spec.seed, "fold": spec.fold,
            "proxy": spec.proxy, "sparsity_weight": spec.sparsity_weight}

# Sanity: the child must rebuild the same run_id the parent planned. A mismatch
# here means a spec field is not crossing the JSON boundary, and the store would
# fill up with runs that never resume.
for grid, spec in PLAN[:1] + PLAN[-1:]:
    from tools.experiment_grid import RunSpec as _RS
    assert _RS(**spec_json(spec)).run_id == spec.run_id, (grid, spec.run_id)
print(f"wrote {OUT_DIR}/_worker.py; run_id round-trip ok")

In [ ]:
# =====================================================================
# RUN - the workers, in parallel, one subprocess per run.
# =====================================================================
import threading, time

started = time.time()
budget_s = WALL_BUDGET_H * 3600
lock = threading.Lock()
timed_out, failed, finished = [], [], []

def run_one(worker, grid, spec):
    out = store_dir(worker, grid)
    Path(out).mkdir(parents=True, exist_ok=True)
    payload_path = f"{OUT_DIR}/_chunk_w{worker}.json"
    Path(payload_path).write_text(json.dumps({
        "config": config_kwargs(grid), "commit_sha": ENV.get("commit_sha", "unknown"),
        "out_dir": out, "specs": [spec_json(spec)]}))
    env = dict(os.environ)
    if N_GPU:
        # One GPU per worker. The child sees a single device and calls it cuda:0,
        # so nothing in the repo has to know about multi-GPU.
        env["CUDA_VISIBLE_DEVICES"] = str(worker % N_GPU)
    t0 = time.time()
    try:
        proc = subprocess.run([sys.executable, f"{OUT_DIR}/_worker.py", REPO_DIR,
                               payload_path], timeout=PER_RUN_TIMEOUT_S,
                              capture_output=True, text=True, env=env)
        ok = spec.run_id in ResultsStore(out).existing_run_ids()
        return ("ok" if ok else "no-row", time.time() - t0,
                (proc.stdout or "")[-600:] + (proc.stderr or "")[-600:])
    except subprocess.TimeoutExpired:
        return ("timeout", time.time() - t0, "")

def worker_loop(worker):
    for grid, spec in ASSIGNED[worker]:
        elapsed = time.time() - started
        if elapsed > budget_s:
            with lock:
                print(f"[w{worker}] wall budget reached ({elapsed/3600:.2f} h) - stopping",
                      flush=True)
            return
        status, secs, tail = run_one(worker, grid, spec)
        with lock:
            print(f"[w{worker}] {status:<8}{secs:7.0f}s  {grid}/{spec.run_id}"
                  f"   ({elapsed/3600:.2f} h elapsed, "
                  f"{len(finished) + len(timed_out) + len(failed) + 1}/{len(todo)})",
                  flush=True)
            if status == "ok":
                finished.append(spec.run_id)
            elif status == "timeout":
                timed_out.append(spec.run_id)
                quarantined[spec.run_id] = quarantined.get(spec.run_id, 0) + 1
                QUARANTINE.write_text(json.dumps(quarantined, indent=1))
            else:
                failed.append(spec.run_id)
                print(f"        {tail.strip()[:500]}", flush=True)

threads = [threading.Thread(target=worker_loop, args=(w,), daemon=True)
           for w in range(WORKERS)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"\n{'=' * 64}")
print(f"elapsed   : {(time.time() - started) / 3600:.2f} h")
print(f"finished  : {len(finished)}")
print(f"timed out : {len(timed_out)}  {timed_out[:5]}")
print(f"failed    : {len(failed)}  {failed[:5]}")

In [ ]:
# =====================================================================
# MERGE + SUMMARY + ZIP
#
# The per-worker stores become one store per grid, through the same merger the
# repo uses - it checks the header against RUN_COLUMNS and refuses a duplicate
# run_id, so a session run twice cannot double-count a run.
# =====================================================================
from collections import defaultdict

sys.path.insert(0, REPO_DIR)
from tools.merge_results import merge

for grid in GRIDS:
    dest = Path(OUT_DIR) / grid
    dest.mkdir(parents=True, exist_ok=True)
    for w in range(WORKERS):
        src = Path(store_dir(w, grid))
        if (src / "runs.csv").exists() and src.resolve() != dest.resolve():
            merge(src, dest)
    store = ResultsStore(str(dest))
    store.write_env({**ENV, "device": DEVICE, "workers": WORKERS,
                     "grid_config": config_kwargs(grid),
                     "note": "A8 backfill of the runs dropped by the Modal "
                             "stall timeout; merge into results/%s" % grid})
    rows = store.read_runs()
    ok = [r for r in rows if r.get("status") == "ok"]
    print(f"\n{grid}: {len(ok)}/{len(rows)} ok")
    by = defaultdict(list)
    for r in ok:
        by[(r["dataset"], r["gamma"])].append(r)
    for (ds, g), rs in sorted(by.items()):
        accs = [float(r["test_acc"]) for r in rs if r.get("test_acc")]
        cells = [float(r["num_cells"]) for r in rs if r.get("num_cells")]
        print(f"   {ds:<22}g={g:<6}n={len(rs):<3}"
              f"acc={sum(accs)/len(accs):.4f}  "
              f"mean cells={sum(cells)/len(cells) if cells else 0:.1f}")
    bad = [r for r in rows if r.get("status") != "ok"]
    for r in bad[:10]:
        print(f"   FAILED {r['run_id']}: {str(r.get('error'))[:90]}")

# Zipped from OUT_DIR so the archive holds `tnn/` and `tnn_mu0/` at its root,
# which is the shape the merge commands below expect.
archive = shutil.make_archive("/kaggle/working/dmd_gridf_a8", "zip", root_dir=OUT_DIR)
print(f"\nwrote {archive}")
print("""
IF IT DID NOT FINISH
--------------------
1. "Save Version" -> this notebook's output becomes a dataset.
2. New session: "+ Add Data" -> Your Datasets -> that output.
3. Set RESUME_FROM to its path. Everything stored is skipped.

BACK IN THE REPO
----------------
unzip dmd_gridf_a8.zip -d /tmp/gridf
python3 tools/merge_results.py /tmp/gridf/tnn      results/tnn      --dry-run
python3 tools/merge_results.py /tmp/gridf/tnn_mu0  results/tnn_mu0  --dry-run
# then the same two without --dry-run, then:
cp results/tnn/runs.csv     paper/results/tnn/runs.csv
cp results/tnn_mu0/runs.csv paper/results/tnn_mu0/runs.csv
cd paper && python3 scripts/summarize_runs.py && python3 scripts/make_macros.py

The merged grids will span more than one commit (the stored runs are at 4678107
/ f5092c4, these are at HEAD). The diff between them touches recording and
analysis only - collect_graph_samples, the A^col metric, the mu axis, the store
schema - and not the model or the training loop, so the arms stay comparable.
Say so in the paper rather than leaving a reader to find two SHAs in a table.
""")